# 📊 Quantile-on-Quantile Granger Causality — Pure Python

A **full Python reimplementation** of the `mqqcause` R package (v1.0.0).  
**No R required. No rpy2. Runs entirely in Python.**

Based on:
- Sim & Zhou (2015) — QQ regression framework *(Journal of Banking & Finance)*
- Troster (2018) — Quantile Granger causality & Sup-Wald test *(Econometric Reviews)*
- Sinha et al. (2023) — Multivariate conditional extension *(Energy Economics)*

---
| Section | What it covers |
|---------|----------------|
| 1 | Install & import dependencies |
| 2 | Core engine — `gaussian_kernel`, `qq_weights`, `weighted_quantile_regression` |
| 3 | `qq_causality()` — Bivariate QQ Granger Causality |
| 4 | `mqq_causality()` — Multivariate / Conditional |
| 5 | `sup_wald()` — Joint Significance Test |
| 6 | `qq_causality_to_matrix()` — Extract result matrix |
| 7 | Visualisation — Heatmap, 3D Surface, Contour, Significance Heatmap |
| 8 | Full end-to-end example with your own data |

---
## 📦 Section 1 — Install & Import Dependencies

In [1]:
# Install required packages (skipped if already installed)
import subprocess, sys

for pkg in ["numpy", "scipy", "pandas", "plotly"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

print("✅ All packages installed")


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


✅ All packages installed


In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
from dataclasses import dataclass
from typing import Dict, Optional
import warnings
warnings.filterwarnings("ignore")

print("✅ Imports successful")

✅ Imports successful


---
## ⚙️ Section 2 — Core Engine (Building Blocks)

Three low-level functions that power everything above — equivalent to  
`gaussian_kernel()`, `qq_weights()`, and `weighted_qr()` in the R package.

### 2.1 — `gaussian_kernel(u)` — Standard Gaussian PDF

In [3]:
def gaussian_kernel(u: np.ndarray) -> np.ndarray:
    """
    Standard Gaussian kernel: K(u) = phi(u) — the standard normal PDF.

    Parameters
    ----------
    u : array of values to evaluate the kernel at

    Returns
    -------
    np.ndarray of kernel values
    """
    return stats.norm.pdf(u)


# Quick test
u_test = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=float)
print("u      :", u_test)
print("K(u)   :", np.round(gaussian_kernel(u_test), 5))

u      : [-3. -2. -1.  0.  1.  2.  3.]
K(u)   : [0.00443 0.05399 0.24197 0.39894 0.24197 0.05399 0.00443]


### 2.2 — `qq_weights(x, tau, h, cdf_based)` — Kernel weights centred at quantile tau

In [4]:
def qq_weights(
    x: np.ndarray,
    tau: float,
    h: float = 0.05,
    cdf_based: bool = True
) -> np.ndarray:
    """
    Gaussian kernel weights for each observation, centred at quantile tau of x.

    Parameters
    ----------
    x         : 1-D numeric array — the cause variable
    tau       : float in (0,1) — quantile level to centre the kernel on
    h         : bandwidth on the [0,1] CDF scale (default 0.05)
    cdf_based : True = use empirical CDF distances (recommended, matches R package)

    Returns
    -------
    weights : np.ndarray — non-negative, sums to n
    """
    n = len(x)
    if cdf_based:
        ranks = stats.rankdata(x) / n     # map each obs to [0,1] via empirical CDF
        u = (ranks - tau) / h
    else:
        q_tau = np.quantile(x, tau)
        u = (x - q_tau) / (h * np.std(x))

    raw = gaussian_kernel(u)
    return raw / raw.sum() * n           # normalise so weights sum to n


# Quick test
np.random.seed(42)
x_test = np.random.randn(50)
w_test = qq_weights(x_test, tau=0.5)
print(f"n={len(w_test)} | sum(weights)={w_test.sum():.4f}  (should equal n)")

n=50 | sum(weights)=50.0000  (should equal n)


### 2.3 — `weighted_quantile_regression(y, X, tau, weights)` — Core regression

In [5]:
def weighted_quantile_regression(
    y: np.ndarray,
    X: np.ndarray,
    tau: float,
    weights: Optional[np.ndarray] = None
) -> Dict:
    """
    Weighted quantile regression via iteratively reweighted least squares.
    Equivalent to quantreg::rq.wfit in R.

    Parameters
    ----------
    y       : 1-D response vector
    X       : 2-D design matrix (observations × predictors, intercept in col 0)
    tau     : quantile level in (0, 1)
    weights : optional kernel weights (uniform if None)

    Returns
    -------
    dict with 'coef' (coefficient vector) and 'residuals'
    """
    n, p = X.shape
    w = np.ones(n) if weights is None else weights
    w = w / w.sum()

    # Initialise with weighted OLS
    W = np.diag(np.sqrt(w))
    try:
        coef = np.linalg.lstsq(W @ X, W @ y, rcond=None)[0]
    except np.linalg.LinAlgError:
        return {"coef": np.zeros(p), "residuals": y}

    # IRLS: refine toward quantile loss
    for _ in range(50):
        residuals = y - X @ coef
        abs_res   = np.abs(residuals)
        eps       = np.percentile(abs_res[abs_res > 0], 1) if np.any(abs_res > 0) else 1e-6
        abs_res   = np.maximum(abs_res, eps)
        q_w       = np.where(residuals >= 0, tau, 1 - tau) / abs_res
        combined  = w * q_w
        combined /= combined.sum()
        W2        = np.diag(np.sqrt(combined))
        try:
            coef_new = np.linalg.lstsq(W2 @ X, W2 @ y, rcond=None)[0]
        except np.linalg.LinAlgError:
            break
        if np.max(np.abs(coef_new - coef)) < 1e-8:
            coef = coef_new
            break
        coef = coef_new

    return {"coef": coef, "residuals": y - X @ coef}


# Quick test
X_test = np.column_stack([np.ones(50), x_test])
y_test = 0.5 * x_test + np.random.randn(50) * 0.3
fit_wqr = weighted_quantile_regression(y_test, X_test, tau=0.5, weights=w_test)
print(f"Intercept: {fit_wqr['coef'][0]:.4f} | Slope: {fit_wqr['coef'][1]:.4f}")

Intercept: 0.4541 | Slope: 1.2929


---
## 🅐 Section 3 — Bivariate QQ Granger Causality: `qq_causality()`

Tests whether the **τ-quantile of x_{t−1}** Granger-causes the **θ-quantile of y_t**.

For each `(theta, tau)` cell on the grid:
1. Compute kernel weights centred at `tau` on lagged x
2. Run weighted quantile regression of y on `[1, x_lag, y_lag]` at quantile `theta`
3. Extract slope on `x_lag` → local causal effect
4. Paired bootstrap → SE, t-statistic, p-value

In [6]:
@dataclass
class QQCausalityResult:
    """Container for qq_causality / mqq_causality results."""
    results_df:  pd.DataFrame
    cause_name:  str = "X"
    effect_name: str = "Y"
    n_obs:       int = 0
    n_boot:      int = 200
    kind:        str = "bivariate"

    def __repr__(self):
        sig = self.results_df[self.results_df["p_value"] < 0.05]
        pct = 100 * len(sig) / len(self.results_df)
        return (
            f"QQ Causality Result [{self.kind}]\n"
            f"  Cause  : {self.cause_name}\n"
            f"  Effect : {self.effect_name}\n"
            f"  N obs  : {self.n_obs}\n"
            f"  Grid   : {self.results_df['theta'].nunique()} theta × "
            f"{self.results_df['tau'].nunique()} tau\n"
            f"  Sig cells (p<0.05): {len(sig)}/{len(self.results_df)} ({pct:.1f}%)\n"
            f"  Slope range: [{self.results_df['slope'].min():.4f}, "
            f"{self.results_df['slope'].max():.4f}]"
        )

print("✅ QQCausalityResult defined")

✅ QQCausalityResult defined


In [7]:
def qq_causality(
    x: np.ndarray,
    y: np.ndarray,
    y_quantiles: np.ndarray = None,
    x_quantiles: np.ndarray = None,
    bandwidth: float = 0.05,
    n_boot: int = 200,
    cdf_based_kernel: bool = True,
    cause_name: str = "X",
    effect_name: str = "Y",
    verbose: bool = True,
    seed: int = 42
) -> QQCausalityResult:
    """
    Bivariate Quantile-on-Quantile Granger Causality Test.

    Parameters
    ----------
    x                : 1-D array — CAUSE variable    ← load from your dataset
    y                : 1-D array — EFFECT variable   ← load from your dataset
    y_quantiles      : theta grid, e.g. np.arange(0.1, 1.0, 0.1)
    x_quantiles      : tau   grid, e.g. np.arange(0.1, 1.0, 0.1)
    bandwidth        : kernel bandwidth on [0,1] CDF scale (default 0.05)
    n_boot           : bootstrap replicates (200 standard, 500+ for papers)
    cdf_based_kernel : True = rank-based CDF kernel (recommended)
    cause_name       : label for x in output/plots
    effect_name      : label for y in output/plots
    verbose          : print progress?
    seed             : RNG seed for reproducibility
    """
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x, float), np.asarray(y, float)
    assert len(x) == len(y), "x and y must be the same length"

    if y_quantiles is None: y_quantiles = np.arange(0.05, 1.0, 0.05)
    if x_quantiles is None: x_quantiles = np.arange(0.05, 1.0, 0.05)
    y_quantiles, x_quantiles = np.asarray(y_quantiles), np.asarray(x_quantiles)

    x_lag, y_lag, y_cur = x[:-1], y[:-1], y[1:]
    n = len(y_cur)
    total, done, records = len(y_quantiles) * len(x_quantiles), 0, []

    for theta in y_quantiles:
        for tau in x_quantiles:
            w     = qq_weights(x_lag, tau=tau, h=bandwidth, cdf_based=cdf_based_kernel)
            X_mat = np.column_stack([np.ones(n), x_lag, y_lag])
            fit   = weighted_quantile_regression(y_cur, X_mat, tau=theta, weights=w)
            slope = fit["coef"][1]

            boot_slopes = np.empty(n_boot)
            for b in range(n_boot):
                idx = rng.integers(0, n, size=n)
                fb  = weighted_quantile_regression(y_cur[idx], X_mat[idx],
                                                   tau=theta, weights=w[idx])
                boot_slopes[b] = fb["coef"][1]

            se    = np.std(boot_slopes, ddof=1)
            t_val = slope / se if se > 1e-10 else 0.0
            p_val = float(np.mean(np.abs(boot_slopes) >= np.abs(slope)))

            records.append({"theta": theta, "tau": tau,
                            "slope": slope, "t_value": t_val, "p_value": p_val})
            done += 1
            if verbose and done % max(1, total // 10) == 0:
                print(f"  Progress: {done}/{total} ({100*done//total}%)")

    if verbose:
        print(f"\n✅ Done — {len(y_quantiles)} theta × {len(x_quantiles)} tau grid")

    return QQCausalityResult(pd.DataFrame(records), cause_name, effect_name, n, n_boot, "bivariate")

print("✅ qq_causality() defined")

✅ qq_causality() defined


### 3.1 — Simulate Data & Run

In [8]:
np.random.seed(42)
n = 200
x = np.random.randn(n)
y = 0.3 * np.concatenate([[0], x[:-1]]) + np.random.randn(n) * 0.4

print(f"x: n={len(x)}, mean={x.mean():.3f}")
print(f"y: n={len(y)}, mean={y.mean():.3f}")

x: n=200, mean=-0.041
y: n=200, mean=0.024


In [9]:
fit_bivariate = qq_causality(
    x                = x,                          # REQUIRED ► your cause variable
    y                = y,                          # REQUIRED ► your effect variable
    y_quantiles      = np.arange(0.1, 1.0, 0.1),  # theta grid
    x_quantiles      = np.arange(0.1, 1.0, 0.1),  # tau   grid
    bandwidth        = 0.05,
    n_boot           = 200,
    cdf_based_kernel = True,
    cause_name       = "Oil_Price",
    effect_name      = "Stock_Return",
    verbose          = True,
    seed             = 42
)

  Progress: 8/81 (9%)
  Progress: 16/81 (19%)
  Progress: 24/81 (29%)
  Progress: 32/81 (39%)
  Progress: 40/81 (49%)
  Progress: 48/81 (59%)
  Progress: 56/81 (69%)
  Progress: 64/81 (79%)
  Progress: 72/81 (88%)
  Progress: 80/81 (98%)

✅ Done — 9 theta × 9 tau grid


### 3.2 — Print Results

In [10]:
print(fit_bivariate)

QQ Causality Result [bivariate]
  Cause  : Oil_Price
  Effect : Stock_Return
  N obs  : 199
  Grid   : 9 theta × 9 tau
  Sig cells (p<0.05): 0/81 (0.0%)
  Slope range: [-0.3443, 1.2656]


In [11]:
# Full long-form results table
fit_bivariate.results_df.round(4)

,theta,tau,slope,t_value,p_value
0,0.1,0.1,-0.2628,-0.8380,0.525
1,0.1,0.2,0.8641,1.8239,0.450
2,0.1,0.3,0.2335,0.6423,0.470
3,0.1,0.4,-0.0317,-0.0238,0.950
4,0.1,0.5,0.4712,0.3798,0.655
...,...,...,...,...,...
76,0.9,0.5,0.1238,0.2427,0.810
77,0.9,0.6,0.9878,1.7597,0.535
78,0.9,0.7,-0.2329,-0.2973,0.840
79,0.9,0.8,0.1170,0.3097,0.750


---
## 🅑 Section 4 — Multivariate / Conditional QQ Causality: `mqq_causality()`

Adds a **moderators dict** (`{'name': array}`) and optional **x × Z interaction terms** to the regression.

In [12]:
def mqq_causality(
    x: np.ndarray,
    y: np.ndarray,
    moderators: Dict[str, np.ndarray] = None,
    y_quantiles: np.ndarray = None,
    x_quantiles: np.ndarray = None,
    bandwidth: float = 0.05,
    n_boot: int = 200,
    interactions: bool = True,
    cdf_based_kernel: bool = True,
    cause_name: str = "X",
    effect_name: str = "Y",
    verbose: bool = True,
    seed: int = 42
) -> QQCausalityResult:
    """
    Multivariate (Conditional) QQ Granger Causality Test.

    Parameters
    ----------
    x            : 1-D array — CAUSE variable
    y            : 1-D array — EFFECT variable
    moderators   : dict of {name: array} — conditioning variables Z
                   e.g. {'GDP': gdp_array, 'Rate': rate_array}
                   ← each array comes from your dataset
    y_quantiles  : theta grid
    x_quantiles  : tau grid
    bandwidth    : kernel bandwidth (default 0.05)
    n_boot       : bootstrap replicates
    interactions : True = include x*Z cross-terms (full conditional test)
    cdf_based_kernel : True = rank-based CDF kernel
    cause_name, effect_name, verbose, seed : same as qq_causality()
    """
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x, float), np.asarray(y, float)
    if moderators is None: moderators = {}

    if y_quantiles is None: y_quantiles = np.arange(0.05, 1.0, 0.05)
    if x_quantiles is None: x_quantiles = np.arange(0.05, 1.0, 0.05)
    y_quantiles, x_quantiles = np.asarray(y_quantiles), np.asarray(x_quantiles)

    x_lag = x[:-1]
    y_lag = y[:-1]
    y_cur = y[1:]
    mods_lag = {k: np.asarray(v, float)[:-1] for k, v in moderators.items()}
    n = len(y_cur)
    total, done, records = len(y_quantiles) * len(x_quantiles), 0, []

    for theta in y_quantiles:
        for tau in x_quantiles:
            w    = qq_weights(x_lag, tau=tau, h=bandwidth, cdf_based=cdf_based_kernel)
            cols = [np.ones(n), x_lag, y_lag]
            for z in mods_lag.values():           # add moderators
                cols.append(z)
            if interactions:
                for z in mods_lag.values():        # add x*Z cross-terms
                    cols.append(x_lag * z)
            X_mat = np.column_stack(cols)

            fit   = weighted_quantile_regression(y_cur, X_mat, tau=theta, weights=w)
            slope = fit["coef"][1]

            boot_slopes = np.empty(n_boot)
            for b in range(n_boot):
                idx = rng.integers(0, n, size=n)
                fb  = weighted_quantile_regression(y_cur[idx], X_mat[idx],
                                                   tau=theta, weights=w[idx])
                boot_slopes[b] = fb["coef"][1]

            se    = np.std(boot_slopes, ddof=1)
            t_val = slope / se if se > 1e-10 else 0.0
            p_val = float(np.mean(np.abs(boot_slopes) >= np.abs(slope)))

            records.append({"theta": theta, "tau": tau,
                            "slope": slope, "t_value": t_val, "p_value": p_val})
            done += 1
            if verbose and done % max(1, total // 10) == 0:
                print(f"  Progress: {done}/{total} ({100*done//total}%)")

    if verbose:
        print(f"\n✅ Done — {len(y_quantiles)} theta × {len(x_quantiles)} tau grid")

    kind = f"multivariate (moderators: {list(moderators.keys())})"
    return QQCausalityResult(pd.DataFrame(records), cause_name, effect_name, n, n_boot, kind)

print("✅ mqq_causality() defined")

✅ mqq_causality() defined


### 4.1 — Simulate Multivariate Data

In [13]:
np.random.seed(42)
n  = 200
z1 = np.random.randn(n)   # moderator 1 (e.g. interest rate)
z2 = np.random.randn(n)   # moderator 2 (e.g. exchange rate)
x  = np.random.randn(n)
y  = (0.3 * np.concatenate([[0], x[:-1]]) +
      0.2 * np.concatenate([[0], z1[:-1]]) +
      np.random.randn(n) * 0.4)

print(f"Variables: x, y, z1, z2  |  n = {n}")

Variables: x, y, z1, z2  |  n = 200


### 4.2 — Run `mqq_causality()`

In [14]:
fit_multivariate = mqq_causality(
    x            = x,
    y            = y,
    moderators   = {
        "Z1": z1,   # ← replace with your actual control variables
        "Z2": z2
    },
    y_quantiles      = np.arange(0.1, 1.0, 0.1),
    x_quantiles      = np.arange(0.1, 1.0, 0.1),
    bandwidth        = 0.05,
    n_boot           = 200,
    interactions     = True,   # True = include x*Z cross-terms
    cdf_based_kernel = True,
    cause_name       = "Oil_Price",
    effect_name      = "Stock_Return",
    verbose          = True,
    seed             = 42
)

  Progress: 8/81 (9%)
  Progress: 16/81 (19%)
  Progress: 24/81 (29%)
  Progress: 32/81 (39%)
  Progress: 40/81 (49%)
  Progress: 48/81 (59%)
  Progress: 56/81 (69%)
  Progress: 64/81 (79%)
  Progress: 72/81 (88%)
  Progress: 80/81 (98%)

✅ Done — 9 theta × 9 tau grid


### 4.3 — Print Multivariate Results

In [15]:
print(fit_multivariate)

QQ Causality Result [multivariate (moderators: ['Z1', 'Z2'])]
  Cause  : Oil_Price
  Effect : Stock_Return
  N obs  : 199
  Grid   : 9 theta × 9 tau
  Sig cells (p<0.05): 0/81 (0.0%)
  Slope range: [-0.4188, 1.2917]


---
## 🅒 Section 5 — Sup-Wald Joint Significance Test: `sup_wald()`

In [16]:
def sup_wald(qq_result: QQCausalityResult, alpha: float = 0.05) -> Dict:
    """
    Sup-Wald joint significance test across the full (theta, tau) grid.
    Based on Troster (2018).

    Parameters
    ----------
    qq_result : QQCausalityResult — fitted object
    alpha     : significance level — 0.01, 0.05, or 0.10
    """
    t_vals   = qq_result.results_df["t_value"].values
    sup_stat = float(np.max(t_vals ** 2))
    crit_val = {0.10: 10.0, 0.05: 13.0, 0.01: 17.0}.get(alpha, 13.0)
    reject   = sup_stat > crit_val
    sig_n    = int((qq_result.results_df["p_value"] < alpha).sum())

    result = {
        "sup_wald_statistic":  round(sup_stat, 4),
        "critical_value":      crit_val,
        "alpha":               alpha,
        "reject_H0":           reject,
        "conclusion":          (
            f"Reject H0 at {alpha} — evidence of QQ Granger causality"
            if reject else
            f"Fail to reject H0 at {alpha} — no QQ Granger causality"
        ),
        "significant_cells":   sig_n,
        "total_cells":         len(qq_result.results_df)
    }
    for k, v in result.items():
        print(f"  {k:27s}: {v}")
    return result

print("✅ sup_wald() defined")

✅ sup_wald() defined


In [17]:
print("── Sup-Wald Test (Bivariate) ──")
sw_biv = sup_wald(fit_bivariate, alpha=0.05)

── Sup-Wald Test (Bivariate) ──
  sup_wald_statistic         : 19.15
  critical_value             : 13.0
  alpha                      : 0.05
  reject_H0                  : True
  conclusion                 : Reject H0 at 0.05 — evidence of QQ Granger causality
  significant_cells          : 0
  total_cells                : 81


In [18]:
print("── Sup-Wald Test (Multivariate) ──")
sw_mult = sup_wald(fit_multivariate, alpha=0.05)

── Sup-Wald Test (Multivariate) ──
  sup_wald_statistic         : 16.1724
  critical_value             : 13.0
  alpha                      : 0.05
  reject_H0                  : True
  conclusion                 : Reject H0 at 0.05 — evidence of QQ Granger causality
  significant_cells          : 0
  total_cells                : 81


---
## 🅓 Section 6 — Extract Result Matrix: `qq_causality_to_matrix()`

In [19]:
def qq_causality_to_matrix(qq_result: QQCausalityResult, value: str = "t_value") -> pd.DataFrame:
    """
    Pivot the long-form results into a (theta × tau) matrix.

    Parameters
    ----------
    qq_result : QQCausalityResult
    value     : column to pivot — "t_value", "slope", or "p_value"
    """
    return qq_result.results_df.pivot(index="theta", columns="tau", values=value).round(4)

print("✅ qq_causality_to_matrix() defined")

✅ qq_causality_to_matrix() defined


In [20]:
# T-statistic matrix
t_matrix = qq_causality_to_matrix(fit_bivariate, value="t_value")
print("── T-Value Matrix (rows=theta, cols=tau) ──")
t_matrix

── T-Value Matrix (rows=theta, cols=tau) ──


tau,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9
theta,,,,,,,,,
0.1,-0.8380,1.8239,0.6423,-0.0238,0.3798,-0.4153,-0.5009,0.9198,1.5304
0.2,0.5121,3.1490,0.3378,0.3199,0.1682,0.3751,-0.4288,-0.1807,0.5447
0.3,1.7218,4.3761,1.4423,0.5422,-0.3467,1.7960,0.3368,0.7236,0.9378
0.4,1.0612,3.0368,1.0108,0.4901,0.2282,2.0509,0.4504,0.5856,1.6494
0.5,0.6483,2.3085,1.3668,0.2218,0.4589,1.5991,-0.1810,0.5510,1.6386
0.6,1.4893,2.0703,1.7073,0.5324,0.0767,1.2398,-0.2582,1.4228,1.8092
0.7,0.8598,1.3720,0.9740,0.3486,-0.0311,1.8926,-0.0712,0.8473,1.4791
0.8,0.6084,0.7322,0.7738,0.3106,0.1775,1.1670,0.6394,0.9225,1.0351
0.9,1.2729,-0.0472,1.3817,-0.4134,0.2427,1.7597,-0.2973,0.3097,1.0393


In [21]:
# Slope (regression coefficient) matrix
b_matrix = qq_causality_to_matrix(fit_bivariate, value="slope")
print("── Slope Matrix ──")
b_matrix

── Slope Matrix ──


tau,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9
theta,,,,,,,,,
0.1,-0.2628,0.8641,0.2335,-0.0317,0.4712,-0.3443,-0.3007,0.4505,0.2671
0.2,0.1450,0.8267,0.1423,0.2432,0.1240,0.3075,-0.2271,-0.0809,0.1037
0.3,0.3735,1.0937,0.5843,0.3444,-0.1829,1.1885,0.1531,0.2540,0.2076
0.4,0.2347,0.8586,0.3933,0.2508,0.0944,1.2656,0.1949,0.1768,0.4070
0.5,0.1692,0.7198,0.5723,0.1183,0.1993,0.8287,-0.0727,0.1788,0.4106
0.6,0.3617,0.7019,0.7392,0.3033,0.0302,0.6986,-0.0971,0.4430,0.4252
0.7,0.2347,0.5590,0.5456,0.1715,-0.0138,1.0290,-0.0337,0.3073,0.3911
0.8,0.2041,0.4312,0.6256,0.1577,0.0715,0.6663,0.4026,0.3503,0.3481
0.9,0.5231,-0.0261,1.1793,-0.2726,0.1238,0.9878,-0.2329,0.1170,0.4295


---
## 🎨 Section 7 — Visualisation

All four functions return **interactive Plotly figures** — hover, zoom, rotate.

### 7.1 — Heatmap with Significance Stars: `plot_qq_causality_heatmap()`

In [22]:
def plot_qq_causality_heatmap(
    qq_result: QQCausalityResult,
    value: str = "t_value",
    colorscale: str = "RdBu",
    show_stars: bool = True,
    x_label: str = "X Quantile (tau)",
    y_label: str = "Y Quantile (theta)",
    title: str = None
) -> go.Figure:
    """
    Interactive heatmap of QQ causality results.

    Parameters: value ("t_value"/"slope"), colorscale (any Plotly scale),
    show_stars (True = overlay significance stars), axis labels, title.
    """
    mat   = qq_causality_to_matrix(qq_result, value=value)
    p_mat = qq_causality_to_matrix(qq_result, value="p_value")
    taus, thetas, z = mat.columns.tolist(), mat.index.tolist(), mat.values

    annotations = []
    if show_stars:
        for i, theta in enumerate(thetas):
            for j, tau in enumerate(taus):
                p    = p_mat.values[i, j]
                star = "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.10 else ""))
                if star:
                    annotations.append(dict(x=tau, y=theta, text=star,
                                            showarrow=False, font=dict(color="black", size=10)))

    if title is None:
        title = (f"QQ Granger Causality: {qq_result.cause_name} → {qq_result.effect_name}<br>"
                 f"<sup>Colour: {value} | n_boot={qq_result.n_boot}</sup>")

    fig = go.Figure(go.Heatmap(
        x=taus, y=thetas, z=z, colorscale=colorscale, zmid=0,
        colorbar=dict(title=value),
        hovertemplate=f"tau=%{{x:.2f}}<br>theta=%{{y:.2f}}<br>{value}=%{{z:.4f}}<extra></extra>"
    ))
    fig.update_layout(title=title, xaxis_title=x_label, yaxis_title=y_label,
                      annotations=annotations, template="plotly_dark", width=650, height=550)
    return fig

print("✅ plot_qq_causality_heatmap() defined")

✅ plot_qq_causality_heatmap() defined


In [23]:
p_heat = plot_qq_causality_heatmap(
    qq_result  = fit_bivariate,
    value      = "t_value",      # "t_value" or "slope"
    colorscale = "RdBu",         # any Plotly colour scale
    show_stars = True,           # overlay significance stars
    x_label    = "X Quantile (tau)",
    y_label    = "Y Quantile (theta)",
    title      = "QQ Granger Causality: Oil Price → Stock Return"
)
p_heat.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

### 7.2 — Interactive 3D Surface: `plot_qq_causality_3d()`

In [ ]:
def plot_qq_causality_3d(
    qq_result: QQCausalityResult,
    value: str = "t_value",
    colorscale: str = "Plasma",
    show_contour: bool = True,
    x_label: str = "X Quantile (tau)",
    y_label: str = "Y Quantile (theta)",
    title: str = None
) -> go.Figure:
    """Interactive 3D surface. show_contour=True draws floor projection."""
    mat    = qq_causality_to_matrix(qq_result, value=value)
    taus, thetas, z = mat.columns.tolist(), mat.index.tolist(), mat.values
    if title is None:
        title = f"3D QQ Causality: {qq_result.cause_name} → {qq_result.effect_name} ({value})"

    contours = dict(z=dict(show=show_contour, usecolormap=True, project=dict(z=True)))
    fig = go.Figure(go.Surface(
        x=taus, y=thetas, z=z, colorscale=colorscale,
        contours=contours if show_contour else {},
        hovertemplate=f"tau=%{{x:.2f}}<br>theta=%{{y:.2f}}<br>{value}=%{{z:.4f}}<extra></extra>"
    ))
    fig.update_layout(
        title=title, template="plotly_dark", width=700, height=600,
        scene=dict(xaxis_title=x_label, yaxis_title=y_label, zaxis_title=value)
    )
    return fig

print("✅ plot_qq_causality_3d() defined")

In [ ]:
p_3d = plot_qq_causality_3d(
    qq_result    = fit_bivariate,
    value        = "t_value",
    colorscale   = "Plasma",
    show_contour = True,        # draw contour lines on the floor
    title        = None         # auto-generated
)
p_3d.show()

### 7.3 — Contour Plot: `plot_qq_causality_contour()`

In [ ]:
def plot_qq_causality_contour(
    qq_result: QQCausalityResult,
    value: str = "t_value",
    colorscale: str = "RdBu",
    x_label: str = "X Quantile (tau)",
    y_label: str = "Y Quantile (theta)",
    title: str = None
) -> go.Figure:
    """Filled 2-D contour plot."""
    mat = qq_causality_to_matrix(qq_result, value=value)
    taus, thetas, z = mat.columns.tolist(), mat.index.tolist(), mat.values
    if title is None:
        title = f"Contour: {qq_result.cause_name} → {qq_result.effect_name} ({value})"

    fig = go.Figure(go.Contour(
        x=taus, y=thetas, z=z, colorscale=colorscale,
        contours=dict(showlabels=True, labelfont=dict(size=10, color="white")),
        colorbar=dict(title=value)
    ))
    fig.update_layout(title=title, xaxis_title=x_label, yaxis_title=y_label,
                      template="plotly_dark", width=650, height=550)
    return fig

print("✅ plot_qq_causality_contour() defined")

In [ ]:
p_cont = plot_qq_causality_contour(
    qq_result  = fit_bivariate,
    value      = "t_value",
    colorscale = "RdBu",
    title      = None
)
p_cont.show()

### 7.4 — Significance-Only Heatmap: `plot_significance_heatmap()`

Only shows cells where `p_value < alpha` — non-significant cells are masked grey.

In [ ]:
def plot_significance_heatmap(
    qq_result: QQCausalityResult,
    alpha: float = 0.05,
    colorscale: str = "RdBu"
) -> go.Figure:
    """Heatmap showing only statistically significant (theta, tau) cells."""
    t_mat = qq_causality_to_matrix(qq_result, value="t_value")
    p_mat = qq_causality_to_matrix(qq_result, value="p_value")
    z = t_mat.values.copy().astype(float)
    z[p_mat.values >= alpha] = np.nan   # mask non-significant cells
    n_sig = int(np.sum(~np.isnan(z)))

    fig = go.Figure(go.Heatmap(
        x=t_mat.columns.tolist(), y=t_mat.index.tolist(), z=z,
        colorscale=colorscale, zmid=0, colorbar=dict(title="t-value")
    ))
    fig.update_layout(
        title=f"Significant cells (p<{alpha}): {n_sig}/{z.size}  |  "
              f"{qq_result.cause_name} → {qq_result.effect_name}",
        xaxis_title="X Quantile (tau)", yaxis_title="Y Quantile (theta)",
        template="plotly_dark", width=650, height=550
    )
    return fig

print("✅ plot_significance_heatmap() defined")

In [ ]:
p_sig = plot_significance_heatmap(
    qq_result  = fit_bivariate,
    alpha      = 0.05,       # significance threshold
    colorscale = "RdBu"
)
p_sig.show()

---
## 🚀 Section 8 — Full End-to-End Example with Your Own Data

Replace the file path and column names below with your actual dataset.

In [ ]:
# STEP 1: Load your data
# df = pd.read_csv("your_data.csv")     # ← your file
# df = pd.read_excel("your_data.xlsx")  # ← or Excel

# Using simulated data for demonstration:
df = pd.DataFrame({"cause": x, "effect": y, "control1": z1, "control2": z2})
print(f"Loaded: {len(df)} rows")
df.head()

In [ ]:
# STEP 2: Extract numpy arrays from your columns
X_var  = df["cause"].values      # ← your cause column
Y_var  = df["effect"].values     # ← your effect column
Z1_var = df["control1"].values   # ← control variable 1
Z2_var = df["control2"].values   # ← control variable 2

In [ ]:
# STEP 3: Bivariate QQ causality
result = qq_causality(
    x=X_var, y=Y_var,
    y_quantiles = np.arange(0.1, 1.0, 0.1),
    x_quantiles = np.arange(0.1, 1.0, 0.1),
    bandwidth=0.05, n_boot=200,
    cause_name="Cause", effect_name="Effect", seed=42
)
print(result)

In [ ]:
# STEP 4: Multivariate conditional test
result_m = mqq_causality(
    x=X_var, y=Y_var,
    moderators={"C1": Z1_var, "C2": Z2_var},
    y_quantiles = np.arange(0.1, 1.0, 0.1),
    x_quantiles = np.arange(0.1, 1.0, 0.1),
    bandwidth=0.05, n_boot=200, interactions=True,
    cause_name="Cause", effect_name="Effect", seed=42
)
print(result_m)

In [ ]:
# STEP 5: Sup-Wald
print("── Bivariate ──");   sup_wald(result, alpha=0.05)
print("\n── Multivariate ──"); sup_wald(result_m, alpha=0.05)

In [ ]:
# STEP 6: All four plots
plot_qq_causality_heatmap(result, show_stars=True).show()
plot_qq_causality_3d(result).show()
plot_qq_causality_contour(result).show()
plot_significance_heatmap(result).show()

In [ ]:
# STEP 7: Export to CSV
result.results_df.to_csv("qq_causality_results.csv", index=False)
print("✅ Saved to qq_causality_results.csv")